# PersonalAI — Exploratory Data Analysis (MovieLens 25M)

**Every chart in this notebook answers a stated question.** Computation and plotting logic live in `src/data/eda.py`; this notebook orchestrates and interprets.

Prerequisite: run `python scripts/download_data.py` and `python scripts/preprocess.py` first.

In [1]:
import sys
from pathlib import Path

# Robust project-root resolution: works under nbconvert (cwd = notebooks/),
# VS Code, and plain Jupyter (cwd = repo root) alike.
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    if ROOT.name == "notebooks":
        ROOT = ROOT.parent
    else:
        ROOT = ROOT / "notebooks" / ".."
ROOT = ROOT.resolve()
sys.path.insert(0, str(ROOT))

import json
import pandas as pd

from src.data import eda
from src.data.loader import load_processed
from src.utils.logging_config import configure_logging

configure_logging("INFO")
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

## 0. Data quality report

First: what did the cleaning pipeline actually change? (Counts from `data/processed/quality_report.json` — never hand-waved.)

In [2]:
report = json.loads((ROOT / "data" / "processed" / "quality_report.json").read_text())
print(json.dumps(report, indent=2))

{
  "generated_at": "2026-10-12T00:24:05.909375+00:00",
  "elapsed_seconds": 37.5,
  "tables": {
    "movies": {
      "rows_in": 62423,
      "rows_out": 62423,
      "duplicates_removed": 0,
      "missing_dropped": {},
      "issues": [
        "412 movies without parseable release year"
      ]
    },
    "ratings": {
      "rows_in": 25000095,
      "rows_out": 25000095,
      "duplicates_removed": 0,
      "missing_dropped": {},
      "issues": []
    },
    "tags": {
      "rows_in": 1093360,
      "rows_out": 1093333,
      "duplicates_removed": 2,
      "missing_dropped": {},
      "issues": [
        "25 empty tags removed"
      ]
    },
    "links": {
      "rows_in": 62423,
      "rows_out": 62423,
      "duplicates_removed": 0,
      "missing_dropped": {},
      "issues": []
    },
    "genome": {
      "rows_in": 15584448,
      "rows_out": 15584448,
      "duplicates_removed": 0,
      "missing_dropped": {},
      "issues": []
    }
  },
  "user_activity_outliers": {
  

## 1. Load processed data

In [3]:
movies = load_processed("movies")
ratings = load_processed("ratings")
tags = load_processed("tags")
genome = load_processed("genome_scores")

display(movies.head(3))
display(ratings.head(3))
print(f"movies={len(movies):,}  ratings={len(ratings):,}  "
      f"tags={len(tags):,}  genome_rows(>=0.5)={len(genome):,}")

,movieId,title,genres,release_year,genres_list,n_genres,title_clean
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995,"[Adventure, Animation, Children, Comedy, Fantasy]",5,Toy Story
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995,"[Adventure, Children, Fantasy]",3,Jumanji
2,3,Grumpier Old Men (1995),Comedy|Romance,1995,"[Comedy, Romance]",2,Grumpier Old Men


,userId,movieId,rating,timestamp
0,1,5952,4.000,1147868053
1,1,2012,2.500,1147868068
2,1,2011,2.500,1147868079


movies=59,047  ratings=25,000,095  tags=1,093,333  genome_rows(>=0.5)=615,148


## 2. Movies

**Q: Which genres dominate the catalog?** (Feature engineering: long-tail genres need different handling than dominant ones; also the diversity stage needs to know what 'diverse' means here.)

In [4]:
genre_counts = eda.plot_genre_distribution(movies)
genre_counts.tail(10)

2026-10-12 06:07:37,599 | INFO     | src.data.eda | Figure saved: D:\MLandData\IntelligentHybridRecommendationSystem\docs\figures\01_genre_distribution.png


genres_list
Sci-Fi          3502
Adventure       3868
Crime           5024
Documentary     5453
Horror          5746
Action          6913
Romance         7305
Thriller        8330
Comedy         16051
Drama          24465
Name: count, dtype: int64

**Q: Is the catalog recency-biased?** (Freshness signals must be normalized against catalog composition — a 2020 movie having more ratings than a 1940 movie may just reflect catalog size.)

In [5]:
decades = eda.plot_release_year_distribution(movies)
decades

2026-10-12 06:07:37,767 | INFO     | src.data.eda | Figure saved: D:\MLandData\IntelligentHybridRecommendationSystem\docs\figures\02_release_year_distribution.png


release_year
1870        2
1880        7
1890      108
1900      145
1910      233
1920      535
1930     1674
1940     1958
1950     2568
1960     3267
1970     4410
1980     4873
1990     6375
2000    12435
2010    20083
Name: count, dtype: int64

## 3. Users and ratings

**Q: Do users round to whole stars?** (Half-star ratings are rarer — matters for implicit-feedback weighting and thresholding 'positive' interactions.)

In [6]:
rating_counts = eda.plot_rating_distribution(ratings)
rating_counts

2026-10-12 06:07:38,030 | INFO     | src.data.eda | Figure saved: D:\MLandData\IntelligentHybridRecommendationSystem\docs\figures\03_rating_distribution.png


rating
0.500     393068
1.000     776815
1.500     399490
2.000    1640868
2.500    1262797
3.000    4896928
3.500    3177318
4.000    6639798
4.500    2200539
5.000    3612474
Name: count, dtype: int64

**Q: How unequal is user activity?** (CF coverage depends on this: users with 3 ratings are hard to model; the cold-start strategy in Milestone 7 targets them.)

In [7]:
per_user = eda.plot_user_activity(ratings)
per_user.describe()

2026-10-12 06:07:38,727 | INFO     | src.data.eda | Figure saved: D:\MLandData\IntelligentHybridRecommendationSystem\docs\figures\04_user_activity.png


count   162,541.000
mean        153.808
std         268.048
min          20.000
25%          36.000
50%          71.000
75%         162.000
max      32,202.000
dtype: float64

**Q: What share of ratings comes from active vs. inactive users?** (Decides whether models trained on all users are dominated by a minority.)

In [8]:
activity_split = eda.plot_active_vs_inactive(per_user, threshold=50)
activity_split

2026-10-12 06:07:38,833 | INFO     | src.data.eda | Figure saved: D:\MLandData\IntelligentHybridRecommendationSystem\docs\figures\05_active_vs_inactive.png


{'threshold': 50,
 'share_users_active': 0.6305609046332925,
 'share_ratings_from_active': 0.9242926076880907}

## 4. Relationships

**Q: Do widely-rated movies also get rated higher?** (Informs the Bayesian-smoothed popularity score — and the answer 'yes, weakly' is itself a finding worth writing up.)

In [9]:
movie_stats = ratings.groupby("movieId").agg(
    mean_rating=("rating", "mean"), n_ratings=("rating", "size")
)
eda.plot_popularity_vs_rating(movie_stats)
correlation = movie_stats["mean_rating"].corr(movie_stats["n_ratings"])
print(f"Pearson r(mean_rating, n_ratings) = {correlation:+.3f}")

2026-10-12 06:07:40,200 | INFO     | src.data.eda | Figure saved: D:\MLandData\IntelligentHybridRecommendationSystem\docs\figures\06_popularity_vs_rating.png


Pearson r(mean_rating, n_ratings) = +0.107


**Q: Which genres get the highest mean ratings?** (The user preference model will represent users as genre distributions — we need to know the global baseline they deviate from.)

In [10]:
genre_means = eda.plot_genre_vs_rating(ratings, movies)
genre_means

2026-10-12 06:08:38,115 | INFO     | src.data.eda | Figure saved: D:\MLandData\IntelligentHybridRecommendationSystem\docs\figures\07_genre_vs_rating.png


genres_list
Horror        3.294
Comedy        3.424
Children      3.433
Action        3.467
Sci-Fi        3.478
Fantasy       3.512
Adventure     3.517
Thriller      3.523
Romance       3.543
Musical       3.555
Western       3.586
IMAX          3.604
Animation     3.615
Mystery       3.670
Drama         3.677
Crime         3.685
Documentary   3.705
War           3.791
Film-Noir     3.926
Name: rating, dtype: float32

**Q: Does movie age predict attention?** (The freshness signal needs a factual basis: is 'newer = more rated' actually true in this catalog?)

In [11]:
decade_medians = eda.plot_year_vs_popularity(ratings, movies)
decade_medians

2026-10-12 06:08:40,053 | INFO     | src.data.eda | Figure saved: D:\MLandData\IntelligentHybridRecommendationSystem\docs\figures\08_year_vs_popularity.png


decade
1,870.000   20.000
1,880.000   14.000
1,890.000    6.000
1,900.000    3.000
1,910.000    4.000
1,920.000    4.000
1,930.000    4.000
1,940.000    4.000
1,950.000    4.000
1,960.000    5.000
1,970.000    4.000
1,980.000    7.000
1,990.000   19.000
2,000.000    9.000
2,010.000    5.000
Name: n_ratings, dtype: float64

# (auto-filled from the executed run of 2026-10-12 — real numbers only)

# | # | Question | Finding | Consequence for the system |
# |---|---|---|---|
# | 1 | Catalog genres | Drama (24,465) and Comedy (16,051) dominate; Sci-Fi is only 3,502 (5.9%) | Genre priors must be catalog-normalized; diversity stage should not over-reward rare genres mechanically |
# | 2 | Recency bias | Catalog is strongly recency-biased: 20,083 movies from the 2010s vs 3,267 from the 1960s | Freshness signal is confounded by catalog composition — normalize per-decade before using |
# | 3 | Rating rounding | Whole stars dominate (4.0 = 6.64M ratings; 5.0 = 3.61M); half-stars rarer | Rating→interaction mapping thresholds at 2.5/3.5/4.5 split mass sensibly |
# | 4 | Activity inequality | Heavy-tailed: median 71 ratings/user, mean 153.8, max 32,202; IQR bound 351, 10% of users above it | CF quality varies wildly by user; report metrics segmented by user history size |
# | 5 | Active/inactive split | 63% of users are active (≥50 ratings) but they account for 92.4% of all ratings | Training is dominated by active users; popularity baselines are robust but personalization must serve the 37% too |
# | 6 | Popularity↔rating | Weak positive: Pearson r = +0.107 between a movie's rating count and its mean rating | Popular ≠ loved, but not independent either — popularity gets a moderate hybrid weight, not a dominant one |
# | 7 | Genre rating baseline | Film-Noir highest (3.93), Horror lowest (3.29); spread is only ~0.6 stars | Genre preference signals are real but subtle; profile model should use deviations from these baselines |
# | 8 | Age↔attention | Median ratings per movie peak in the 1990s (19), collapse for 2010s (5) despite bigger catalog | Newer movies have less interaction data → cold-start item handling matters (M7); recency alone would misrank |
